# MongoDB Landing Zone — Exploration

Verbindung: `liora-vm.matthiaskoehler.com:27017`  
Datenbank: `airline_landing`  

| Collection | Befüllt durch | Inhalt |
|---|---|---|
| `adsb_raw` | `collectors/adsb_collector.py` | ADS-B Snapshots (adsb.lol, Berlin 60 nm Radius) |
| `opensky_raw` | `collectors/opensky_collector.py` | Departures & Arrivals pro Airport (OpenSky Network) |

**Sektionen 1–8:** ADS-B Exploration  
**Sektionen 9–11:** OpenSky Exploration + Cross-Collection Join

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from datetime import timezone

sys.path.insert(0, '.')
from db.mongo.connector import from_env

db = from_env().connect()
print("Verbunden mit MongoDB")

## 1. Collections und Dokumentenanzahl

In [ ]:
client = db._client
database = client["airline_landing"]

print("Collections in airline_landing:\n")
for name in sorted(database.list_collection_names()):
    count = database[name].count_documents({})
    latest = database[name].find_one({}, {"collected_at": 1, "_id": 0}, sort=[("collected_at", -1)])
    latest_ts = latest.get("collected_at", "?")[:19] if latest else "?"
    print(f"  {name:<20}  docs={count:>5}  latest={latest_ts}")

col = db.collection("adsb_raw")

## 2. Neueste Snapshots (ohne `ac`-Array)

In [ ]:
recent = list(col.find({}, {'ac': 0}).sort('collected_at', -1).limit(10))

df_snaps = pd.DataFrame([
    {
        'collected_at': d['collected_at'],
        'total': d['total'],
        'source': d['source'],
        '_id': str(d['_id']),
    }
    for d in recent
])
print(df_snaps.to_string(index=False))

## 3. Letzten Snapshot als DataFrame aufklappen

In [ ]:
latest = col.find_one({}, sort=[('collected_at', -1)])
print(f"Snapshot: {latest['collected_at']}  —  {latest['total']} aircraft")

df = pd.DataFrame(latest['ac'])
print(f"Shape: {df.shape}")
print(f"Spalten: {list(df.columns)}")

In [ ]:
# Lesbare Übersicht
cols = [c for c in ['hex', 'flight', 'r', 't', 'alt_baro', 'gs', 'lat', 'lon', 'dst'] if c in df.columns]
print(df[cols].to_string(index=False))

## 4. Datenqualität — Vollständigkeit der Kernfelder

In [ ]:
key_fields = ['hex', 'flight', 'lat', 'lon', 'alt_baro', 'gs', 'r', 't']
present = {f: df[f].notna().sum() if f in df.columns else 0 for f in key_fields}
for field, count in present.items():
    pct = count / len(df) * 100
    print(f"  {field:<12} {count:>3}/{len(df)}  ({pct:.0f}%)")

## 5. alt_baro: Zahl vs. 'ground'

In [ ]:
if 'alt_baro' in df.columns:
    on_ground = (df['alt_baro'] == 'ground').sum()
    airborne  = (df['alt_baro'] != 'ground').sum()
    print(f"Am Boden:    {on_ground}")
    print(f"In der Luft: {airborne}")

    df['alt_baro_ft'] = pd.to_numeric(df['alt_baro'], errors='coerce')
    df['on_ground']   = df['alt_baro'] == 'ground'

## 6. Top-Callsigns über Berlin

In [ ]:
if 'flight' in df.columns:
    top = (
        df['flight']
        .str.strip()
        .replace('', pd.NA)
        .dropna()
        .value_counts()
        .head(20)
    )
    print(top.to_string())

## 7. Alle Snapshots zusammenführen — Zeitreihe der Flugzeugzahl

In [ ]:
all_snaps = list(col.find({}, {'ac': 0}).sort('collected_at', 1))

df_ts = pd.DataFrame([
    {'collected_at': d['collected_at'], 'total': d['total']}
    for d in all_snaps
])
df_ts['collected_at'] = pd.to_datetime(df_ts['collected_at'], utc=True)

print(df_ts.to_string(index=False))
print(f"\nSchnitt: {df_ts['total'].mean():.1f} aircraft/snapshot")
print(f"Min/Max: {df_ts['total'].min()} / {df_ts['total'].max()}")

---
## 9. OpenSky Raw — Überblick

Jedes Dokument = ein API-Call (departures **oder** arrivals) für einen Airport und ein Zeitfenster.

In [ ]:
from datetime import datetime, timezone

osky_col = db.collection("opensky_raw")

docs = list(osky_col.find({}, {"flights": 0}).sort("collected_at", -1))

df_osky = pd.DataFrame([
    {
        "collected_at": d["collected_at"][:19],
        "endpoint":     d["endpoint"],
        "airport":      d["query_params"]["airport"],
        "flight_count": d["flight_count"],
        "window_h":     round((d["query_params"]["end"] - d["query_params"]["begin"]) / 3600),
        "_id":          str(d["_id"]),
    }
    for d in docs
])

print(f"Dokumente in opensky_raw: {len(df_osky)}\n")
print(df_osky.to_string(index=False))

## 10. OpenSky Flüge aufklappen

Alle Flights aus allen opensky_raw-Dokumenten als flacher DataFrame.

In [ ]:
rows = []
for doc in osky_col.find({}).sort("collected_at", 1):
    for f in doc.get("flights", []):
        rows.append({
            "icao24":    f.get("icao24", ""),
            "callsign":  (f.get("callsign") or "").strip(),
            "dep":       f.get("estDepartureAirport") or "?",
            "arr":       f.get("estArrivalAirport") or "?",
            "firstSeen": datetime.fromtimestamp(f["firstSeen"], timezone.utc).strftime("%Y-%m-%d %H:%M") if f.get("firstSeen") else None,
            "lastSeen":  datetime.fromtimestamp(f["lastSeen"],  timezone.utc).strftime("%H:%M")          if f.get("lastSeen")  else None,
            "endpoint":  doc["endpoint"],
            "airport":   doc["query_params"]["airport"],
        })

df_flights = pd.DataFrame(rows).drop_duplicates(subset=["icao24", "firstSeen"])
print(f"Unique Flüge (dedupliziert): {len(df_flights)}\n")
print(df_flights.head(20).to_string(index=False))

## 11. Cross-Collection Join: ADS-B ↔ OpenSky

Join-Key: `adsb_raw.ac[].hex` = `opensky_raw.flights[].icao24` (ICAO24 Transponderadresse)

- **ADS-B** liefert: Echtzeit-Position, Höhe, Geschwindigkeit, Flugzeugtyp  
- **OpenSky** ergänzt: Abflug- und Zielflughafen, Callsign, Abflugzeit  

Einschränkung: Match nur möglich wenn ein Flugzeug im ADS-B-Snapshot sichtbar war **und** im OpenSky-Zeitfenster erfasst wurde.

In [ ]:
# OpenSky-Index aufbauen: icao24 → beste verfügbare Fluginfo
opensky_index = {}
for doc in osky_col.find({}):
    for f in doc.get("flights", []):
        icao = (f.get("icao24") or "").lower()
        if icao and icao not in opensky_index:
            opensky_index[icao] = {
                "callsign_osky": (f.get("callsign") or "").strip(),
                "dep":           f.get("estDepartureAirport") or "?",
                "arr":           f.get("estArrivalAirport") or "?",
                "firstSeen":     datetime.fromtimestamp(f["firstSeen"], timezone.utc).strftime("%H:%M UTC") if f.get("firstSeen") else "?",
            }

# Neuesten ADS-B Snapshot joinen
latest_adsb = db.collection("adsb_raw").find_one({}, sort=[("collected_at", -1)])
snapshot_time = latest_adsb["collected_at"][:19]

matches = []
for ac in latest_adsb.get("ac", []):
    hex_id = (ac.get("hex") or "").lower()
    if hex_id in opensky_index:
        osky = opensky_index[hex_id]
        matches.append({
            "icao24":    hex_id,
            "callsign":  (ac.get("flight") or "").strip() or osky["callsign_osky"],
            "type":      ac.get("t", "?"),
            "alt_ft":    ac.get("alt_baro", "?"),
            "gs_kts":    round(ac.get("gs") or 0),
            "lat":       round(ac.get("lat", 0), 3) if ac.get("lat") else "?",
            "lon":       round(ac.get("lon", 0), 3) if ac.get("lon") else "?",
            "dep":       osky["dep"],
            "arr":       osky["arr"],
            "firstSeen": osky["firstSeen"],
        })

df_joined = pd.DataFrame(matches).sort_values("callsign")

print(f"ADS-B Snapshot: {snapshot_time}  ({latest_adsb['total']} aircraft)")
print(f"OpenSky icao24s im Index: {len(opensky_index)}")
print(f"Matches: {len(df_joined)}\n")
print(df_joined.to_string(index=False))

## 8. Verbindung schließen

In [ ]:
db.close()
print("Verbindung geschlossen")